## Menghubungkan Google Drive dan Memuat Dataset Kasus

In [1]:


import os
import json
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split

# Mount Google Drive
drive.mount('/content/drive')


BASE_DRIVE_DIR = '/content/drive/MyDrive/CBR/'
PROCESSED_CSV_PATH = os.path.join(BASE_DRIVE_DIR, 'data/processed/cases.csv')
EVAL_DIR = os.path.join(BASE_DRIVE_DIR, 'data/eval/')

os.makedirs(EVAL_DIR, exist_ok=True)

# Load Data
if not os.path.exists(PROCESSED_CSV_PATH):
    print(f"[ERROR] File {PROCESSED_CSV_PATH} tidak ditemukan. Pastikan Tahap 2 Sudah dijalankan.")
else:
    df_cases = pd.read_csv(PROCESSED_CSV_PATH)
    print(f"Sukses memuat data! Total: {len(df_cases)} kasus wanprestasi siap diproses.")

Mounted at /content/drive
Sukses memuat data! Total: 30 kasus wanprestasi siap diproses.


## Pembagian Data (Splitting Data)

In [2]:

df_train, df_test = train_test_split(df_cases, test_size=0.20, random_state=42)

print(f"Hasil Pembagian Data (Rasio 80:20):")
print(f"- Jumlah Data Train (Case Base) : {len(df_train)} dokumen")
print(f"- Jumlah Data Test (Kasus Baru) : {len(df_test)} dokumen")

# Menampilkan ID Kasus yang masuk ke dalam data test untuk bahan pengujian nanti
print(f"- ID Kasus untuk Uji Ulang      : {df_test['case_id'].tolist()}")

Hasil Pembagian Data (Rasio 80:20):
- Jumlah Data Train (Case Base) : 24 dokumen
- Jumlah Data Test (Kasus Baru) : 6 dokumen
- ID Kasus untuk Uji Ulang      : [31, 19, 27, 21, 12, 13]


## Vektorisasi dan Fungsi Retrieval

In [3]:


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Inisialisasi TfidfVectorizer dengan stop_words bahasa Indonesia (bawaan sederhana)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit & Transform pada data train (Membangun matriks vektor Case Base)
tfidf_matrix_train = tfidf_vectorizer.fit_transform(df_train['ringkasan_fakta'].fillna(''))

def retrieve(query: str, k: int = 5):
    """
    Fungsi Retrieval mendeteksi top-k kasus terdekat berdasarkan query teks baru.
    """
    # Langkah 1: Pre-process & ubah query teks baru menjadi vektor TF-IDF
    query_vector = tfidf_vectorizer.transform([query])

    # Langkah 2: Hitung cosine-similarity antara vektor query dengan seluruh matriks train
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix_train).flatten()

    # Langkah 3: Ambil indeks top-k dengan skor kemiripan tertinggi
    top_k_indices = argsort_scores = np.argsort(similarity_scores)[::-1][:k]

    # Langkah 4: Kembalikan daftar dictionary berisi case_id dan skor kemiripannya
    results = []
    for idx in top_k_indices:
        results.append({
            "case_id": int(df_train.iloc[idx]['case_id']),
            "no_perkara": df_train.iloc[idx]['no_perkara'],
            "similarity_score": float(similarity_scores[idx])
        })
    return results

print("Vektorisasi TF-IDF Selesai! Fungsi retrieve() siap digunakan.")

Vektorisasi TF-IDF Selesai! Fungsi retrieve() siap digunakan.


## Skenario Pengujian Dokumen

In [4]:

eval_queries_list = []

print("Membuat skenario pengujian dokumen (Ground-Truth)...")
for i, row in df_test.head(5).iterrows():
    # Mengonversi ke string terlebih dahulu dan memberikan fallback jika data kosong/NaN
    fakta_text = str(row['ringkasan_fakta']) if pd.notna(row['ringkasan_fakta']) else "Gugatan wanprestasi atas pemenuhan kewajiban perjanjian."

    query_text = fakta_text[:500]

    query_entry = {
        "query_id": int(row['case_id']),
        "query_text": query_text,
        "ground_truth_case_id": [int(row['case_id'])]
    }
    eval_queries_list.append(query_entry)

# Simpan skenario ke format JSON
queries_json_path = os.path.join(EVAL_DIR, 'queries.json')
with open(queries_json_path, 'w', encoding='utf-8') as json_file:
    json.dump(eval_queries_list, json_file, indent=4, ensure_ascii=False)

print(f"File Skenario Evaluasi Berhasil Disimpan di: {queries_json_path}\n" + "="*60)

# Uji coba langsung fungsi retrieve
sample_query = eval_queries_list[0]["query_text"]
print("Menjalankan Test Retrieval pada salah satu query uji:")
print(f"Query: {sample_query[:150]}...")
print("-" * 40)

retrieved_cases = retrieve(sample_query, k=5)
for rank, case in enumerate(retrieved_cases, 1):
    print(f"Rank {rank} -> Case ID: {case['case_id']} | Skor Kemiripan: {case['similarity_score']:.4f} | No: {case['no_perkara']}")

Membuat skenario pengujian dokumen (Ground-Truth)...
File Skenario Evaluasi Berhasil Disimpan di: /content/drive/MyDrive/CBR/data/eval/queries.json
Menjalankan Test Retrieval pada salah satu query uji:
Query: : , tanggal 25 september 2024 yangamarnya berbunyi sebagai berikut:hal 2 dari 8 putusan no 825/pdt/2024/pt sby putusan.mahkamahagung.go.id mahkamah ag...
----------------------------------------
Rank 1 -> Case ID: 10 | Skor Kemiripan: 0.6251 | No: 296/PDT/2021/PT
Rank 2 -> Case ID: 25 | Skor Kemiripan: 0.6086 | No: 740/PDT/2021/PT
Rank 3 -> Case ID: 32 | Skor Kemiripan: 0.5905 | No: 858/PDT/2024/PT
Rank 4 -> Case ID: 24 | Skor Kemiripan: 0.5692 | No: 692/PDT/2024/PT
Rank 5 -> Case ID: 30 | Skor Kemiripan: 0.4059 | No: 804/PDT/2021/PT
